# LLM Referral Toolkit Log Preparation

This script converts the raw ChatGPT logs (e.g., 'conversations-000.json') into level 1 turn logs (minimally cleaned logs, e.g.,
'log_level1.csv'), then applies PII redaction (Part 4) before anything leaves this notebook. Each row corresponds to a user or AI message ("turn"). The rows are in chronological order of the messages in the log.

**PII redaction is active.** The unredacted `level1_logs/` files are an intermediate step that exist locally in the Colab session  only until Part 4 to read and redact them. They are never actually uploaded to Box. Only `redacted_logs/` (the PII-redacted version) is uploaded.


Within root_file_path (the folder this script is in):
* 01_llm_preprocessing_colab.py        <- this script
* logs/

  * raw_logs/

      * P1_conversations-000.json      <- input: standard ChatGPT export format
      * P1_conversations-001.json      <- same participant
      * P2_conversations.json          <- smaller account may only have a single file
      ...
  * level1_logs/ <= INTERMEDIATE ONLY, not persisted to Box; stays empty in the Box folder

      * P1_log_level1.csv              <- ONE merged file per participant that is input to Part 4 and never uploaded
      * P2_log_level1.csv
        * Its columns:
          * turn_id (str), a unique identifier for this turn. Build as "{participant_id}_{conversation_id}\_{turn_index}".
          * timestamp (timestamp)
          * level1event (str), e.g. 'user-sends-message' or 'assistant-sends-message'. Could extend later to measure different events besides turns.
          * timestamp_type (str),  which is either 'start', 'end', or 'instant'. Here, "instant" means this level‑1 event occurs at a single timestamp; "start" and "end" indicate this event spans a duration, and the timestamp marks its beginning or end. Currently turns are always "instant," occurring at a single timestamp
          * actor (str), which is either 'user' or 'AI'
          * message_content (str), the text content of this turn message
          * conversation_snapshot (list of tuples, each tuple is (actor, message content)): the full chronological conversation history up to and including this turn. May be useful if later signal needs prior-turn context. Set to off by default.

  * redacted_logs/ <= ACTUAL OUTPUT of this notebook

      * P1_log_level1.csv              <- PII-redacted version of the level 1 log; same as above, with detected entities replaced by placeholder tokens. This is what 02_llm_compute_metrics_meta.ipynb takes as input.
      * ...

  * pii_audit/

      * P1_log_level1_pii_audit.csv    <- one row per flagged entity

  * session_metadata.csv <= metadata file
      * If the folder contains ChatGPT export logs, specify each participant's ID in the 'participant_id' column and each raw file's name in the 'raw_filename' column. Each row represents a raw log file, so a  participant with multiple numbered export files will have multiple rows with the same participant_id. Upload raw logs in \<participant_id\>_conversations[NNN].json format and this notebook will perform the conversion to level1 logs and redaction.

  * signals_config.csv <= NOT used by this notebook; read by 02_llm_compute_metrics_meta.ipynb
      * One row per (construct, signal, method) combination. Columns: construct, signal, method, params_key, active. Determines which signals/methods actually get scored.

  * signal_params.json <= NOT used by this notebook; read by 02_llm_compute_metrics_meta.ipynb
      * Keyed by params_key (matches signals_config.csv). Holds each signal's anchor_items (CCR), key_phrases (lexicon), prototype_sentences (SBERT), and definition (zero-shot).

*Note: Open.AI provides multiple raw log files if a user has a longer Chatlog history.*


In [ ]:
# drive.flush_and_unmount()

In [ ]:
import sys
!{sys.executable} -m pip uninstall boxsdk box-sdk-gen -y --quiet
!{sys.executable} -m pip install "boxsdk==3.9.2" --quiet

In [ ]:
from boxsdk import Client, CCGAuth
print("success")

In [ ]:
import sys
!{sys.executable} -m pip install boxsdk

In [ ]:
import json
import os

import pandas as pd
import re
from pathlib import Path

from google.colab import userdata
from boxsdk import Client, OAuth2

access_token = userdata.get('BOX_DEVELOPER_TOKEN')
auth = OAuth2(client_id=None, client_secret=None, access_token=access_token)
client = Client(auth)

me = client.user().get()
print(f"Authenticated as: {me.name} ({me.login})")

## Part 1. Specify Relevant Information
Specify box and gdrive file locations:

In [ ]:
# Box IDs
box_raw_logs_folder_id = "401719399046"
box_level1_logs_folder_id = "401728187423"
box_session_metadata_file_id = "2360310440998"
box_redacted_logs_folder_id = "401874734353"

# Local paths of Colab's Disk: not saved, wiped after every runtime
root_file_path = "/content/logs"
raw_logs_dir = Path(root_file_path) / "raw_logs"
level1_logs_dir = Path(root_file_path) / "level1_logs"
session_metadata_path = Path(root_file_path) / "session_metadata.csv"

raw_logs_dir.mkdir(parents=True, exist_ok=True) # create directories
level1_logs_dir.mkdir(parents=True, exist_ok=True)

# Dowload session_metadata and raw logs
with open(session_metadata_path, 'wb') as f:
    client.file(box_session_metadata_file_id).download_to(f)

for item in client.folder(box_raw_logs_folder_id).get_items():
    local_path = raw_logs_dir / item.name
    with open(local_path, 'wb') as f:
        client.file(item.id).download_to(f)
    print(f"  downloaded {item.name}")

In [ ]:
print(os.listdir(raw_logs_dir))
print(os.listdir(root_file_path))  # confirms session_metadata.csv landed too

Specify what raw log data we want to keep

In [ ]:
keep_roles = {'user', 'assistant'}
actor_labels = {'user': 'user', 'assistant': 'AI'} # relabel ChatGPT's names for roles
include_conversation_snapshot = False # Snapshot feature that'll take a snapshot fo the entire log history until a present turn

## Part 2. Parse Raw Chatlog data


Load 'leve1actions.csv', which specifies how to convert among raw logs, our modified raw logs, and level 1 logs.

In [ ]:
# Checks that loaded JSON is valid list of conversation objects with
# expected keys (mapping, current_node, conversation_id)
def validate_export_format(data, filename):
    """
    Guards against the file-path/wrong-format bug from the old pipeline:
    fail loudly and specifically instead of silently loading something
    that isn't actually a ChatGPT conversations export.
    """
    if not isinstance(data, list):
        raise ValueError(
            f"{filename}: expected a top-level LIST of conversation objects "
            f"(standard ChatGPT export format), got {type(data).__name__}. "
            f"Check you're pointing at 'conversations.json', not another "
            f"export file (e.g. 'user.json', 'message_feedback.json')."
        )
    if len(data) == 0: # empty export
        raise ValueError(f"{filename}: conversation list is empty.")
    sample = data[0]
    required_keys = {'mapping', 'current_node', 'conversation_id'}
    missing = required_keys - set(sample.keys())
    if missing:
        raise ValueError(
            f"{filename}: first conversation object is missing expected "
            f"key(s) {missing}. This doesn't look like a standard ChatGPT "
            f"export — check the file wasn't partially downloaded or "
            f"re-exported in a different format."
        )

# Turns ChatGPT's conversation tree into a chronological list of conversations
def linearize_conversation(conversation):
    """
    Walk the mapping tree from current_node back to the root via parent
    pointers, then reverse to get chronological node order for the
    currently-active branch (i.e. ignoring abandoned edit branches).
    """
    mapping = conversation.get('mapping', {})
    current_node_id = conversation.get('current_node')
    if current_node_id is None or current_node_id not in mapping:
        return []

    nodes_in_order = []
    node_id = current_node_id
    seen = set()
    while node_id is not None:
        if node_id in seen:
            break
        seen.add(node_id)
        node = mapping.get(node_id)
        if node is None:
            break
        nodes_in_order.append(node)
        node_id = node.get('parent')

    nodes_in_order.reverse()
    return nodes_in_order

# Get text (from user/chatbot)
def extract_text(message):
    """
    message['content']['parts'] is usually a list of strings, but can
    contain non-text parts (e.g. image references as dicts) for
    multimodal turns. Keep string parts only; skip non-text silently
    but track how often this happens (see main loop) so it's visible
    rather than silently lossy.
    """
    content = message.get('content', {}) or {}
    parts = content.get('parts', [])
    text_parts = [p for p in parts if isinstance(p, str)]
    dropped_nontext = len(parts) - len(text_parts)
    text = '\n'.join(t for t in text_parts if t.strip())
    return text, dropped_nontext

## Part 3. Convert raw logs to level 1 logs
This step generates a level1actions_df
- **turn_id** (str), a unique identifier for this turn.
- **timestamp** (timestamp)
- **level1event** (str), e.g. 'user-sends-message' or 'assistant-sends-message'
- **timestamp_type** (str), either 'start', 'end', or 'instant'; instant" means this level‑1 event occurs at a single timestamp;  
- **actor** (str), e.g. 'user' or 'AI'
- **message_content** (str), text content of this turn message
- **conversation_snapshot** (list of tuples), each tuple is the full chronological conversation history up to and including this turn. Default is off.

### Extraction function

In [ ]:
# Turn JSON into df of conversations
def extract_turns_from_file(json_path, participant_id):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    validate_export_format(data, json_path.name) # check validity of file

    rows = []
    total_dropped_nontext = 0
    source_file = json_path.name

    # Look at all vonversations
    for conversation in data:
        conversation_id = conversation.get('conversation_id', 'unknown')
        conversation_title = conversation.get('title', '') or ''
        nodes = linearize_conversation(conversation)

        turn_index = 0
        snapshot_so_far = []  # list of (actor, message_content) tuples, in order

        # Go through ChatGPT nodes and get messages
        for node in nodes:
            message = node.get('message')
            if message is None:
                continue  # root/empty nodes have no message

            author = message.get('author', {}) or {}
            role = author.get('role')
            if role not in keep_roles: # only get user/AI messages
                continue

            text, dropped = extract_text(message)
            total_dropped_nontext += dropped
            if not text.strip():
                continue  # skip empty turns (e.g. pure image messages)

            create_time = message.get('create_time')
            timestamp = (
                pd.to_datetime(create_time, unit='s', utc=True)
                if create_time is not None else pd.NaT
            )

            actor = actor_labels[role]
            snapshot_so_far.append((actor, text))

            turn_id = f"{participant_id}_{conversation_id}_{turn_index}"
            row = {
                'turn_id': turn_id,
                'timestamp': timestamp,
                'level1event': f'{actor}-sends-message',
                'timestamp_type': 'instant',
                'actor': actor,
                'message_content': text,
                'participant_id': participant_id,
                'conversation_id': conversation_id,
                'conversation_title': conversation_title,
                'turn_index': turn_index,
                'word_count': len(text.split()),
                'char_count': len(text),
                'source_file': source_file,
            }
            if include_conversation_snapshot:
                # copy the current list; snapshot up to AND
                # including this turn, matching backstage_chat_snapshot's
                # "snapshot at this timestamp" semantics
                row['conversation_snapshot'] = json.dumps(list(snapshot_so_far))

            rows.append(row)
            turn_index += 1

    # Build df from plain list of dicts
    if total_dropped_nontext:
        print(f"  [note] {json_path.name}: dropped {total_dropped_nontext} "
              f"non-text content part(s) (likely images/attachments).")

    # Converts list of row dicts into a DataFrame ONCE per file
    return pd.DataFrame(rows)


### Process Participant Logs

In [ ]:
def main():
    metadata = pd.read_csv(session_metadata_path, dtype=str) # read everything as a string (keeps 007 instead of convert to 7)
    required_cols = {'participant_id', 'raw_filename'}
    missing = required_cols - set(metadata.columns)
    if missing:
        raise ValueError(f"session_metadata.csv is missing column(s): {missing}")

    summary_rows = []

    # Group so each participant's numbered files are processed together.
    # Files aren't expected to overlap (each numbered export file covers a
    # disjoint set of conversations), so processing order shouldn't matter
    # much but list them in a consistent order anyway for reproducible
    # output.
    for participant_id, group in metadata.groupby('participant_id', sort=False):
        raw_filenames = group['raw_filename'].tolist()
        print(f"Processing {participant_id} ({len(raw_filenames)} raw file(s))...")

        file_dfs = []
        for raw_filename in raw_filenames:
            raw_path = raw_logs_dir / raw_filename
            if not raw_path.exists():
                print(f"  [SKIP] file not found: {raw_path}")
                continue
            file_dfs.append(extract_turns_from_file(raw_path, participant_id))

        if not file_dfs:
            print(f"  [SKIP] {participant_id}: no valid raw files found.")
            continue

        combined = pd.concat(file_dfs, ignore_index=True)

        # De-duplicate turns that appear in more than one export file
        # (overlapping conversations from repeated exports). Keep the
        # first occurrence per the file order above.
        n_before = len(combined)
        combined = combined.drop_duplicates(subset='turn_id', keep='first')
        n_dropped = n_before - len(combined)
        if n_dropped:
            print(f"  [note] dropped {n_dropped} duplicate turn(s) found "
                  f"across overlapping export files.")

        combined = combined.sort_values('timestamp').reset_index(drop=True)

        out_path = level1_logs_dir / f"{participant_id}_log_level1.csv"
        combined.to_csv(out_path, index=False)

        n_conv = combined['conversation_id'].nunique() if len(combined) else 0
        n_user = (combined['actor'] == 'user').sum() if len(combined) else 0
        n_ai = (combined['actor'] == 'AI').sum() if len(combined) else 0
        date_min = combined['timestamp'].min() if len(combined) else None
        date_max = combined['timestamp'].max() if len(combined) else None

        print(f"  -> {len(combined)} turns across {n_conv} conversations "
              f"({n_user} user / {n_ai} AI), "
              f"{date_min} to {date_max}")

        summary_rows.append({
            'participant_id': participant_id,
            'n_raw_files': len(raw_filenames),
            'n_duplicate_turns_dropped': n_dropped,
            'n_turns': len(combined),
            'n_conversations': n_conv,
            'n_user_turns': n_user,
            'n_ai_turns': n_ai,
            'date_min': date_min,
            'date_max': date_max,
        })

    # Write one summary CSV covering all participants and return summary df
    summary_df = pd.DataFrame(summary_rows)
    summary_path = level1_logs_dir / '_preprocessing_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f"\nDone. Summary written to {summary_path}")
    return summary_df


if __name__ == '__main__':
    summary = main()
    summary

# Preview resulting dataframe
preview_pid = summary['participant_id'].iloc[0]
preview_df = pd.read_csv(level1_logs_dir / f'{preview_pid}_log_level1.csv')
preview_df.head(10)

## Part 4. Redact Personally Identifiable Information


* Applies automated detection and redaction to level-1 turn
logs before they enter the signal-scoring pipeline.

* Runs Microsoft Presidio (used in [Fang et al., 2025](https://arxiv.org/abs/2602.18415)) over user and AI turns. Replaces detected entities with placeholder tokens (<PERSON>, <REDACTED>, ...) and writes the redacted log to redacted_logs/.

* Writes a per-participant file to pii_audit/, recording every flagged item (entity type, detected text, confidence score, character offsets) for manual review.

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Reuse paths already defined earlier in this notebook — not new/hardcoded ones
input_dir = level1_logs_dir
redacted_dir = Path(root_file_path) / "redacted_logs"
audit_dir = Path(root_file_path) / "pii_audit"
redacted_dir.mkdir(exist_ok=True)
audit_dir.mkdir(exist_ok=True)

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# What to redact
entities = [
    "PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION",
    "US_SSN", "CREDIT_CARD", "DATE_TIME", "URL"
]

operators = {
    "DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"}),
    "PERSON": OperatorConfig("replace", {"new_value": "<PERSON>"}),
}

def redact_turn(text: str, turn_id: str):
    """Applies Presidio to a single turn. Returns (redacted_text, audit_rows)."""
    if not isinstance(text, str) or not text.strip():
        return text, []

    results = analyzer.analyze(text=text, entities=entities, language="en")
    anonymized = anonymizer.anonymize(text=text, analyzer_results=results, operators=operators)

    audit_rows = [
        {
            "turn_id": turn_id,
            "entity_type": r.entity_type,
            "detected_text": text[r.start:r.end],  # kept ONLY in audit file for manual review
            "score": r.score,
            "start": r.start,
            "end": r.end,
        }
        for r in results
    ]
    return anonymized.text, audit_rows

def redact_participant_log(level1_csv_path: Path):
    df = pd.read_csv(level1_csv_path)
    all_audit_rows = []

    redacted_texts = []
    for _, row in df.iterrows():
        redacted_text, audit_rows = redact_turn(row["message_content"], row["turn_id"])
        redacted_texts.append(redacted_text)
        all_audit_rows.extend(audit_rows)

    df["message_content"] = redacted_texts  # fixed: was "text", actual column is "message_content"

    out_path = redacted_dir / level1_csv_path.name
    df.to_csv(out_path, index=False)

    audit_df = pd.DataFrame(all_audit_rows)
    audit_path = audit_dir / f"{level1_csv_path.stem}_pii_audit.csv"
    audit_df.to_csv(audit_path, index=False)

    return out_path, audit_path, len(all_audit_rows)

# Run over all participant level1 logs
for csv_path in sorted(input_dir.glob("*_level1.csv")):  # narrower pattern — skips _preprocessing_summary.csv
    df_check = pd.read_csv(csv_path, nrows=1)
    if "message_content" not in df_check.columns:
        print(f"  [skip] {csv_path.name} — missing 'message_content' column, not a level1 log")
        continue
    out_path, audit_path, n_flags = redact_participant_log(csv_path)
    print(f"{csv_path.name}: {n_flags} entities flagged -> {out_path.name}")

# Upload redacted output back to Box — NOT the unredacted level1 files
# Update in place if the file already exists, otherwise create it
existing_items = {
    item.name: item.id
    for item in client.folder(box_redacted_logs_folder_id).get_items()
}

for csv_file in redacted_dir.glob("*.csv"):
    if csv_file.name in existing_items:
        client.file(existing_items[csv_file.name]).update_contents(str(csv_file))
        print(f"  updated {csv_file.name}")
    else:
        client.folder(box_redacted_logs_folder_id).upload(str(csv_file))
        print(f"  uploaded {csv_file.name}")

In [ ]:
print(list(input_dir.glob("*.csv"))) # sanity check for debugging